In [ ]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)


Mounted at /content/drive


In [ ]:
#labels = ['Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration','No Finding', 'Mass', 'Nodule', 'Pneumonia', 'Pneumothorax', 'Consolidation', 'Edema', 'Emphysema', 'Fibrosis', 'Pleural_Thickening', 'Hernia']

In [ ]:
import pandas as pd

df= pd.read_csv('/content/drive/MyDrive/ChestXray_Dataset/final_1.csv')

In [ ]:
import pandas as pd

# Assuming df is your original dataframe with a 'labels' column

# Step 1: Filter rows containing "No Finding" in the labels column
no_finding_rows = df[df['Finding Labels'].str.contains('No Finding', na=False)].sample(n=4160)

# Step 2: Filter rows containing "Pneumonia" in the labels column
pneumonia_rows = df[df['Finding Labels'].str.contains('Pneumonia', na=False)].sample(n=4160)

# Step 3: Combine the two filtered dataframes
combined_df = pd.concat([no_finding_rows, pneumonia_rows], ignore_index=True)

# Now you have 2048 rows in total with 1024 of each label
print(combined_df.shape)


(8320, 3)


In [ ]:
import os
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Define device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
final = combined_df
#class_names = set(labels)
label_map = {class_name: idx for idx, class_name in enumerate(final['Finding Labels'].unique())}

In [ ]:
num_classes=len(label_map)

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),   # Resize to 224x224 (like ResNet expects)
    transforms.ToTensor(),           # Convert image to tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Normalize with ImageNet stats
])


In [ ]:
import torch
from torch.utils.data import Dataset
from PIL import Image

class CustomImageDatasetFromDF(Dataset):
    def __init__(self, dataframe, label_map, transform=None):
        """
        Args:
            dataframe (pd.DataFrame): DataFrame containing image paths and labels.
            label_map (dict): A dictionary mapping string labels to numeric indices.
            transform (callable, optional): Optional transform to be applied on an image sample.
        """
        self.data = dataframe
        self.transform = transform
        self.label_mapping = label_map

    def __len__(self):
        # Return the total number of images
        return len(self.data)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        # Get the image path from the 'full_image_path' column
        img_path = self.data.iloc[idx]['full_image_path']
        image = Image.open(img_path).convert("RGB")  # Open and convert to RGB

        # Get the label from the DataFrame (assuming it's in a 'Finding Labels' column)
        label = self.data.iloc[idx]['Finding Labels']

        # If it's a multi-label classification problem, assume labels are pipe-separated
        if isinstance(label, str):
            label = label.split('|')  # Split multiple labels

        # Initialize a zero tensor of size equal to the number of labels (15 in this case)
        numeric_labels = torch.zeros(len(self.label_mapping), dtype=torch.float32)

        # Set the corresponding index to 1 for each label present
        for lbl in label:
            numeric_labels[self.label_mapping[lbl]] = 1

        # Apply transformations if provided
        if self.transform:
            image = self.transform(image)

        return image, numeric_labels


In [ ]:
# Define the dataset instance using the updated DataFrame with image paths
dataset = CustomImageDatasetFromDF(dataframe=final,label_map =label_map, transform=transform)

# Create a DataLoader for batching
train_loader = DataLoader(dataset, batch_size=32, shuffle=True,num_workers=2)


# **ResNet**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Define a basic residual block
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.downsample = downsample  # Downsample layer for matching dimensions of residual connection

    def forward(self, x):
        residual = x  # Save the input (skip connection)

        out = F.relu(self.bn1(self.conv1(x)))  # First convolution
        out = self.bn2(self.conv2(out))  # Second convolution

        if self.downsample:
            residual = self.downsample(x)  # Apply downsampling if needed

        out += residual  # Add the skip connection
        out = F.relu(out)  # Apply ReLU after the addition
        return out


In [ ]:
class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes=15):  # `layers` defines the number of blocks per layer
        super(ResNet, self).__init__()
        self.in_channels = 64

        # Initial Convolution + BatchNorm + ReLU
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        # ResNet layers
        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)

        # Fully connected layer
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, block, out_channels, blocks, stride=1):
        downsample = None
        if stride != 1 or self.in_channels != out_channels:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

        layers = []
        layers.append(block(self.in_channels, out_channels, stride, downsample))
        self.in_channels = out_channels
        for _ in range(1, blocks):
            layers.append(block(out_channels, out_channels))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))  # Initial convolution
        x = self.maxpool(x)

        # Pass through each layer
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        # Global average pooling and fully connected layer
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)

        return x


In [ ]:
#configuration = [2,2,2,2] # resnet 18
#configuration = [3, 4, 6, 3] # resnet 50
configuration = [3, 4, 23, 3] # resnet 101

In [ ]:
def ResNet18(configuration,num_classes):
    return ResNet(ResidualBlock,configuration , num_classes=num_classes)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
# Instantiate the model
model = ResNet18(configuration,num_classes).to(device)

# Define loss and optimizer
criterion = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)




# **DenseNet**

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class Bottleneck(nn.Module):
    def __init__(self, in_channels, growth_rate):
        super(Bottleneck, self).__init__()
        inner_channels = 4 * growth_rate
        self.bottleneck = nn.Sequential(
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, inner_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(inner_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(inner_channels, growth_rate, kernel_size=3, padding=1, bias=False)
        )

    def forward(self, x):
        return torch.cat([x, self.bottleneck(x)], 1)

class Transition(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(Transition, self).__init__()
        self.transition = nn.Sequential(
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.AvgPool2d(kernel_size=2, stride=2)
        )

    def forward(self, x):
        return self.transition(x)

class DenseNet(nn.Module):
    def __init__(self, block, nblocks, growth_rate=32, reduction=0.5, num_classes=2):
        super(DenseNet, self).__init__()
        self.growth_rate = growth_rate
        inner_channels = 2 * growth_rate

        # Initial convolution
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, inner_channels, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(inner_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )

        # Dense blocks
        self.features = nn.Sequential()
        num_channels = inner_channels
        for i, num_layers in enumerate(nblocks):
            self.features.add_module(f'dense_block_{i}', self._make_dense_block(block, num_layers, num_channels))
            num_channels += num_layers * growth_rate
            if i != len(nblocks) - 1:
                out_channels = int(num_channels * reduction)
                self.features.add_module(f'transition_{i}', Transition(num_channels, out_channels))
                num_channels = out_channels

        # Final layers
        self.bn = nn.BatchNorm2d(num_channels)
        self.relu = nn.ReLU(inplace=True)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(num_channels, num_classes)

    def _make_dense_block(self, block, num_layers, in_channels):
        layers = []
        for i in range(num_layers):
            layers.append(block(in_channels, self.growth_rate))
            in_channels += self.growth_rate
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.features(x)
        x = self.bn(x)
        x = self.relu(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

def densenet121(num_classes=2):
    return DenseNet(Bottleneck, [6, 12, 24, 16], growth_rate=32, num_classes=num_classes)

In [ ]:
model = densenet121(num_classes=2).to(device)
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
num_epochs = 25
i = 0
from datetime import datetime
from torch.cuda.amp import GradScaler, autocast

scaler = GradScaler()



<ipython-input-14-16e740f9bd25>:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [ ]:
# Example of a simple training loop with the new dataset
for epoch in range(num_epochs):
    model.train()
    for images, labels in train_loader:
        # Move data to the device (GPU or CPU)
        images, labels = images.to(device), labels.to(device)
        print("we were here")
        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        print('step done at',datetime.now().strftime('%H:%M:%S'),'number', i)
        i += 1
        if i % 1000 == 0:
            print('we are in ',i)


    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item()}')


we were here


<ipython-input-15-80e2f9707453>:8: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


step done at 13:16:27 number 0
we were here
step done at 13:16:27 number 1
we were here
step done at 13:16:53 number 2
we were here
step done at 13:16:54 number 3
we were here
step done at 13:17:20 number 4
we were here
step done at 13:17:20 number 5
we were here
step done at 13:17:46 number 6
we were here
step done at 13:17:47 number 7
we were here
step done at 13:18:14 number 8
we were here
step done at 13:18:14 number 9
we were here
step done at 13:18:41 number 10
we were here
step done at 13:18:41 number 11
we were here
step done at 13:19:09 number 12
we were here
step done at 13:19:09 number 13
we were here
step done at 13:19:35 number 14
we were here
step done at 13:19:35 number 15
we were here
step done at 13:20:03 number 16
we were here
step done at 13:20:03 number 17
we were here
step done at 13:20:31 number 18
we were here
step done at 13:20:31 number 19
we were here
step done at 13:21:00 number 20
we were here
step done at 13:21:00 number 21
we were here
step done at 13:21:2

In [ ]:
torch.save(model.state_dict(), 'resnet18-version2.0.pth')

NameError: name 'torch' is not defined

In [ ]:
train = {}
for images, labels in train_loader:
  train[images]=labels

KeyboardInterrupt: 

In [ ]:
for images, labels in train_loader:
    print(labels)  # Unpack only the first two elements if there are more
    print(type(labels))
    break



tensor([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0.,

In [ ]:
final_labels= valid_images_df_13
final_labels['full_image_path'] = final_labels['full_image_path'].apply(lambda x: x.replace('images_01', 'images_13'))
valid_images_df_13=final_labels

In [ ]:
final.to_csv('final_1.csv', index=False)


In [ ]:
test = pd.read_csv('/content/valid_images_df_08.csv')
total = len(test)
j = 0

In [ ]:
import torch
from PIL import Image
from torchvision import transforms
i = 0
for image,labels,path in zip(test['Image Index'],test['Finding Labels'],test['full_image_path']):
  if labels == 'Pneumonia':

    img_path = path.replace('images_01','images_08')
    image = Image.open(img_path).convert("RGB")

    # Preprocess the image
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    input_image = transform(image).unsqueeze(0)  # Add batch dimension
    input_image = input_image.to(device)
    # Load your model (assuming it's already trained)
    model.eval()

    # Perform inference
    with torch.no_grad():
        output = model(input_image)
        output = torch.sigmoid(output)  # Apply sigmoid to the logits

    # Make binary predictions
    predictions = (output > 0.5).float()
    i +=1
    predicted_classes = [class_name for idx, class_name in enumerate(label_map) if predictions[0][idx] == 1]
    print(predicted_classes)

['Pneumonia']
['No Finding']
['Pneumonia']


In [ ]:
j

1

In [ ]:
net = SimpleCNN()

criterion = nn.CrossEntropyLoss()


optimizer = optim.Adam(net.parameters(), lr=0.001)

In [ ]:
# Set device to GPU if available, otherwise fallback to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move the model to the device (GPU or CPU)
net.to(device)

In [ ]:
# Training the model
num_epochs = 10

for epoch in range(num_epochs):
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        inputs, labels = data
        inputs, labels = inputs.cuda(), labels.cuda()
        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = net(inputs)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        # Print statistics
        running_loss += loss.item()
        if i % 100 == 99:
            print(f'Epoch {epoch + 1}, Batch {i + 1}, Loss: {running_loss / 100:.4f}')
            running_loss = 0.0

print('Finished Training')


In [ ]:
correct = 0
total = 0

with torch.no_grad():
    for data in testloader:
        images, labels = data
        images, labels = inputs.cuda(), labels.cuda()
        outputs = net(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the network on the 10,000 test images: {100 * correct / total:.2f}%')
